In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.config import *
from src.utils import setup_logger, chunk_text, load_jsonl
from sentence_transformers import SentenceTransformer
import torch
import numpy as np
import pickle
from tqdm import tqdm

logger = setup_logger()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Setup complete")
print(f"Device: {device}")

Setup complete
Device: cuda


In [2]:
print("=== Loading Processed Documents ===\n")

metadata_file = PROCESSED_DATA_DIR / "ingestion_metadata.jsonl"
all_documents = load_jsonl(metadata_file)

print(f"✓ Loaded {len(all_documents)} documents")
print(f"✓ Total characters: {sum(doc['char_count'] for doc in all_documents):,}")

=== Loading Processed Documents ===

✓ Loaded 1675 documents
✓ Total characters: 777,751


In [3]:
print("=== Chunking Documents ===\n")
print(f"Chunk size: {CHUNK_SIZE} characters")
print(f"Chunk overlap: {CHUNK_OVERLAP} characters\n")

all_chunks = []

for doc in tqdm(all_documents, desc="Chunking documents"):
    text = doc['text']
    chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)
    
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'chunk_id': f"{doc.get('filename', 'doc')}_{i}",
            'text': chunk,
            'source': doc['source'],
            'chunk_index': i,
            'original_doc': doc.get('filename', 'unknown')
        })

print(f"\n✓ Created {len(all_chunks)} chunks from {len(all_documents)} documents")
print(f"✓ Average chunks per document: {len(all_chunks) / len(all_documents):.1f}")

=== Chunking Documents ===

Chunk size: 512 characters
Chunk overlap: 50 characters



Chunking documents: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1675/1675 [00:00<00:00, 370404.34it/s]


✓ Created 2908 chunks from 1675 documents
✓ Average chunks per document: 1.7


In [4]:
print("=== Loading Embedding Model ===\n")
print(f"Model: {EMBEDDING_MODEL_NAME}")
print("Loading model to GPU...\n")

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

print(f"✓ Model loaded on {device}")
print(f"✓ Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

=== Loading Embedding Model ===

Model: sentence-transformers/all-MiniLM-L6-v2
Loading model to GPU...



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\patel\Documents\chitraksha\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\patel\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Model loaded on cuda
✓ Embedding dimension: 384


In [5]:
print("=== Generating Embeddings ===\n")
print(f"Total chunks to process: {len(all_chunks)}")
print("This will take a few minutes...\n")

batch_size = 32
all_embeddings = []

for i in tqdm(range(0, len(all_chunks), batch_size), desc="Generating embeddings"):
    batch = all_chunks[i:i+batch_size]
    texts = [chunk['text'] for chunk in batch]
    
    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=False
    )
    
    all_embeddings.extend(embeddings)

all_embeddings = np.array(all_embeddings)

print(f"\n✓ Generated {len(all_embeddings)} embeddings")
print(f"✓ Embedding shape: {all_embeddings.shape}")
print(f"✓ Memory usage: {all_embeddings.nbytes / 1024 / 1024:.2f} MB")

=== Generating Embeddings ===

Total chunks to process: 2908
This will take a few minutes...



Generating embeddings: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 91/91 [00:02<00:00, 35.64it/s]


✓ Generated 2908 embeddings
✓ Embedding shape: (2908, 384)
✓ Memory usage: 4.26 MB


In [6]:
print("=== Saving Chunks and Embeddings ===\n")

chunks_file = PROCESSED_DATA_DIR / "chunks.pkl"
embeddings_file = PROCESSED_DATA_DIR / "embeddings.npy"

with open(chunks_file, 'wb') as f:
    pickle.dump(all_chunks, f)

np.save(embeddings_file, all_embeddings)

print(f"✓ Chunks saved to: {chunks_file}")
print(f"✓ Embeddings saved to: {embeddings_file}")
print(f"\nFile sizes:")
print(f"  Chunks: {chunks_file.stat().st_size / 1024 / 1024:.2f} MB")
print(f"  Embeddings: {embeddings_file.stat().st_size / 1024 / 1024:.2f} MB")

=== Saving Chunks and Embeddings ===

✓ Chunks saved to: C:\Users\patel\Documents\chitraksha\data\processed\chunks.pkl
✓ Embeddings saved to: C:\Users\patel\Documents\chitraksha\data\processed\embeddings.npy

File sizes:
  Chunks: 0.94 MB
  Embeddings: 4.26 MB


In [10]:
print("=== Testing Embedding Quality ===\n")

test_query = "I feel anxious and can't sleep"
query_embedding = embedding_model.encode([test_query], convert_to_numpy=True)[0]

from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

similarities = [cosine_similarity(query_embedding, emb) for emb in all_embeddings]
top_5_indices = np.argsort(similarities)[-5:][::-1]

print(f"Query: '{test_query}'")
print(f"\nTop 5 most relevant chunks:\n")

for rank, idx in enumerate(top_5_indices, 1):
    chunk = all_chunks[idx]
    score = similarities[idx]
    print(f"{rank}. Similarity: {score:.3f}")
    print(f"   Source: {chunk['source']}")
    print(f"   Text: {chunk['text'][:150]}...")
    print()

=== Testing Embedding Quality ===

Query: 'I feel anxious and can't sleep'

Top 5 most relevant chunks:

1. Similarity: 0.563
   Source: hf_chatbot
   Text: . Manage Stress and Worries: If you find yourself lying in bed with racing thoughts, consider keeping a journal nearby to jot down your worries. It ca...

2. Similarity: 0.563
   Source: hf_chatbot
   Text: . Manage Stress and Worries: If you find yourself lying in bed with racing thoughts, consider keeping a journal nearby to jot down your worries. It ca...

3. Similarity: 0.559
   Source: hf_chatbot
   Text: or staying asleep. 8. Avoidance of triggers or situations that provoke anxiety. Anxiety attacks can be triggered by specific stressors or occur withou...

4. Similarity: 0.559
   Source: hf_chatbot
   Text: or staying asleep. 8. Avoidance of triggers or situations that provoke anxiety. Anxiety attacks can be triggered by specific stressors or occur withou...

5. Similarity: 0.533
   Source: hf_chatbot
   Text: anxiety attacks

In [11]:
print("=== Embedding Statistics ===\n")

print(f"Total chunks: {len(all_chunks)}")
print(f"Total embeddings: {len(all_embeddings)}")
print(f"Embedding dimension: {all_embeddings.shape[1]}")
print(f"\nChunks by source:")

source_counts = {}
for chunk in all_chunks:
    source = chunk['source']
    source_counts[source] = source_counts.get(source, 0) + 1

for source, count in sorted(source_counts.items()):
    percentage = (count / len(all_chunks)) * 100
    print(f"  {source}: {count} chunks ({percentage:.1f}%)")

print(f"\nAverage chunk length: {np.mean([len(c['text']) for c in all_chunks]):.0f} characters")
print(f"Median chunk length: {np.median([len(c['text']) for c in all_chunks]):.0f} characters")

=== Embedding Statistics ===

Total chunks: 2908
Total embeddings: 2908
Embedding dimension: 384

Chunks by source:
  hf_chatbot: 992 chunks (34.1%)
  kaggle_mental_health: 1386 chunks (47.7%)
  pdf: 530 chunks (18.2%)

Average chunk length: 288 characters
Median chunk length: 253 characters


In [12]:
print("=== Data Quality Verification ===\n")

empty_chunks = [c for c in all_chunks if not c['text'].strip()]
duplicate_chunks = len(all_chunks) - len(set(c['text'] for c in all_chunks))
zero_embeddings = np.sum(np.all(all_embeddings == 0, axis=1))

print(f"Empty chunks: {len(empty_chunks)}")
print(f"Duplicate chunks: {duplicate_chunks}")
print(f"Zero embeddings: {zero_embeddings}")

if empty_chunks or zero_embeddings > 0:
    print("\n⚠ Warning: Data quality issues detected")
else:
    print("\n✓ All chunks and embeddings are valid")

print(f"\n✓ Ready for vector store indexing")

=== Data Quality Verification ===

Empty chunks: 0
Duplicate chunks: 1265
Zero embeddings: 0

✓ All chunks and embeddings are valid

✓ Ready for vector store indexing


In [14]:
import json
from datetime import datetime

summary = {
    "processing_timestamp": datetime.now().isoformat(),
    "total_documents": len(all_documents),
    "total_chunks": len(all_chunks),
    "embedding_dimension": int(all_embeddings.shape[1]),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "chunks_by_source": source_counts,
    "avg_chunk_length": int(np.mean([len(c['text']) for c in all_chunks])),
}

summary_file = PROCESSED_DATA_DIR / "chunking_summary.json"
with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print("=== Processing Summary Saved ===")
print(f"Location: {summary_file}")
print("\nSummary:")
print(json.dumps(summary, indent=2))

=== Processing Summary Saved ===
Location: C:\Users\patel\Documents\chitraksha\data\processed\chunking_summary.json

Summary:
{
  "processing_timestamp": "2025-12-07T14:05:53.159382",
  "total_documents": 1675,
  "total_chunks": 2908,
  "embedding_dimension": 384,
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "chunk_size": 512,
  "chunk_overlap": 50,
  "chunks_by_source": {
    "hf_chatbot": 992,
    "kaggle_mental_health": 1386,
    "pdf": 530
  },
  "avg_chunk_length": 287
}


In [15]:
print("="*60)
print("PHASE 3: CHUNKING & EMBEDDINGS COMPLETE")
print("="*60)

print("\n✓ Documents chunked into optimal sizes")
print(f"✓ {len(all_chunks):,} chunks created")
print(f"✓ {len(all_embeddings):,} embeddings generated")
print(f"✓ Chunks saved to: chunks.pkl")
print(f"✓ Embeddings saved to: embeddings.npy")
print(f"✓ Summary saved to: chunking_summary.json")

print("\n" + "="*60)
print("READY FOR PHASE 4: VECTOR STORE")
print("="*60)

print("\nNext steps:")
print("1. Save this notebook (Ctrl+S)")
print("2. Create new notebook: 04_vector_store.ipynb")
print("3. We'll build FAISS index")
print("4. Implement semantic search")
print("5. Test retrieval quality")

PHASE 3: CHUNKING & EMBEDDINGS COMPLETE

✓ Documents chunked into optimal sizes
✓ 2,908 chunks created
✓ 2,908 embeddings generated
✓ Chunks saved to: chunks.pkl
✓ Embeddings saved to: embeddings.npy
✓ Summary saved to: chunking_summary.json

READY FOR PHASE 4: VECTOR STORE

Next steps:
1. Save this notebook (Ctrl+S)
2. Create new notebook: 04_vector_store.ipynb
3. We'll build FAISS index
4. Implement semantic search
5. Test retrieval quality
